In [ ]:
import numpy as np
import pandas as pd

from functools import partial
import importlib

import src.data
import src.features
import src.kernel_normalisation
import src.kernels
import src.krr
import src.metrics
import src.multikernel

from src.data import encode_labels, load_data, train_val_split
from src.features import DenseSIFT, HOG
from src.kernel_normalisation import unit_diag
from src.kernels import *
from src.krr import KernelRidgeRegression
from src.metrics import accuracy
from src.multikernel import *

In [2]:
DATA_DIR = "data/"

X_tr, X_te, y_tr_raw = load_data(DATA_DIR)

y_tr, inv_map = encode_labels(y_tr_raw)

n_classes = len(np.unique(y_tr))

### SIFT features

In [60]:
dsift = DenseSIFT(patch_size=16, stride=8, num_cells=2, num_angles=8).fit(X_tr)

X_tr_sift = dsift.transform(X_tr)
X_te_sift = dsift.transform(X_te)

X_train_sift, X_val_sift, y_train_sift, y_val_sift = train_val_split(
    X_tr_sift, y_tr, test_size=0.3, seed=20
)

In [61]:
X_train_sift.shape

(3500, 864)

In [4]:
hog = HOG(
    image_shape=(3, 32, 32),
    colour_mode="rgb",
    cell_size=8,
    block_size=2,
    block_stride=1,
    num_bins=6,
    signed=False,
).fit(X_tr)

X_tr_hog = hog.transform(X_tr)
X_te_hog = hog.transform(X_te)

X_train_hog, X_val_hog, y_train_hog, y_val_hog = train_val_split(
    X_tr_hog, y_tr, test_size=0.3, seed=20
)

In [62]:
gammas_sift = estimate_chi2_gammas_channel(X_train_sift)
sift_chi2 = partial(chi2_rbf_kernel_matrix_channel, gammas=gammas_sift)

gammas_hog = estimate_chi2_gammas_channel(X_train_hog)
hog_chi2 = partial(chi2_rbf_kernel_matrix_channel, gammas=gammas_hog)

# Two-kernel setup:
specs: list[KernelSpec] = [
    {
        "name": "sift_chi2",
        "Z": X_train_sift,
        "kernel_fn": sift_chi2,
        "diag_fn": unit_diag,
        "normalise": True,
    },
    {
        "name": "hog_chi2",
        "Z": X_train_hog,
        "kernel_fn": hog_chi2,
        "diag_fn": unit_diag,
        "normalise": True,
    },
]

out = beta_grid_search_cv_two_kernels(
    specs,
    y_train_sift,
    n_classes=n_classes,
    model="krr",
    betas=np.array(
        [
            0.2,
            0.25,
            0.3,
            0.35,
            0.4,
            0.45,
            0.5,
            0.55,
            0.6,
            0.65,
            0.7,
            0.75,
            0.8,
            0.85,
            0.9,
            0.95,
        ]
    ),
    k=5,
    seed=20,
    lam=1e-4,
    verbose=1,
)

print("best_beta:", out["best_beta"], "best_mean_acc:", out["best_mean_acc"])

beta=0.200 mean_acc=0.5600
beta=0.250 mean_acc=0.5614
beta=0.300 mean_acc=0.5626
beta=0.350 mean_acc=0.5634
beta=0.400 mean_acc=0.5623
beta=0.450 mean_acc=0.5611
beta=0.500 mean_acc=0.5603
beta=0.550 mean_acc=0.5611
beta=0.600 mean_acc=0.5614
beta=0.650 mean_acc=0.5591
beta=0.700 mean_acc=0.5569
beta=0.750 mean_acc=0.5551
beta=0.800 mean_acc=0.5543
beta=0.850 mean_acc=0.5546
beta=0.900 mean_acc=0.5489
beta=0.950 mean_acc=0.5466
best_beta: 0.3499999940395355 best_mean_acc: 0.5634285714285714


## After tuning parameters, train the best model.

### Check the best model

In [63]:
beta = float(np.asarray(out["best_beta"]).item())

# IMPORTANT: use the partial kernels you defined (with gammas)
Ks_tr = sift_chi2(X_train_sift, X_train_sift)
Kh_tr = hog_chi2(X_train_hog, X_train_hog)

# Match CV: unit-diagonal normalisation per kernel
Ks_tr = normalise_train_gram(Ks_tr, unit_diag(X_train_sift))
Kh_tr = normalise_train_gram(Kh_tr, unit_diag(X_train_hog))

Ktr = beta * Ks_tr + (1.0 - beta) * Kh_tr

Ks_val = sift_chi2(X_val_sift, X_train_sift)
Kh_val = hog_chi2(X_val_hog, X_train_hog)

Ks_val = normalise_cross_gram(Ks_val, unit_diag(X_val_sift), unit_diag(X_train_sift))
Kh_val = normalise_cross_gram(Kh_val, unit_diag(X_val_hog), unit_diag(X_train_hog))

K_val = beta * Ks_val + (1.0 - beta) * Kh_val

best_model = KernelRidgeRegression(n_classes=n_classes, kernel_fn=None, lam=1e-4).fit(
    None, y_train_sift, K=Ktr
)
yhat_val, _ = best_model.predict(K_star=K_val)
print("val acc:", accuracy(y_val_hog, yhat_val))

val acc: 0.5753333333333334


### Train on full dataset for submission

In [ ]:
beta = float(np.asarray(out["best_beta"]).item())

# Re-estimate gammas on FULL training set (not the CV subset)
gammas_sift = estimate_chi2_gammas_channel(X_tr_sift)
sift_chi2 = partial(chi2_rbf_kernel_matrix_channel, gammas=gammas_sift)

gammas_hog = estimate_chi2_gammas_channel(X_tr_hog)
hog_chi2 = partial(chi2_rbf_kernel_matrix_channel, gammas=gammas_hog)

# FULL train Gram
Ks_tr = sift_chi2(X_tr_sift, X_tr_sift)
Kh_tr = hog_chi2(X_tr_hog, X_tr_hog)

Ks_tr = normalise_train_gram(Ks_tr, unit_diag(X_tr_sift))
Kh_tr = normalise_train_gram(Kh_tr, unit_diag(X_tr_hog))

Ktr = beta * Ks_tr + (1.0 - beta) * Kh_tr

best_model = KernelRidgeRegression(n_classes=n_classes, kernel_fn=None, lam=1e-4).fit(
    None, y_tr, K=Ktr
)

# TEST x TRAIN cross Gram
Ks_te_tr = sift_chi2(X_te_sift, X_tr_sift)
Kh_te_tr = hog_chi2(X_te_hog, X_tr_hog)

Ks_te_tr = normalise_cross_gram(Ks_te_tr, unit_diag(X_te_sift), unit_diag(X_tr_sift))
Kh_te_tr = normalise_cross_gram(Kh_te_tr, unit_diag(X_te_hog), unit_diag(X_tr_hog))

K_star = beta * Ks_te_tr + (1.0 - beta) * Kh_te_tr

yte_int, _ = best_model.predict(K_star=K_star)
yte = np.array([inv_map[i] for i in yte_int])

sub = pd.DataFrame({"Prediction": yte})
sub.index += 1
sub.to_csv("results/submission.csv", index_label="Id")